# 03 — Feature provenance, preprocessing, and split design

**Purpose:** create a reproducible, leakage-aware preprocessing and evaluation plan.

This notebook intentionally does not fit a model. First resolve the label-generation process and complete the feature decision register in Notebook 02.

## 1. Load data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Put the CSV beside this notebook, or change DATA_PATH to its full path.
DATA_PATH = Path("metaverse_transactions_dataset(1).csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the original CSV beside this notebook "
        f"or edit DATA_PATH. CSV files in this folder: {[p.name for p in candidates]}"
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 2. Declare target and candidate feature decisions

In [ ]:
TARGET = "anomaly"

# Conservative provisional exclusions. Review these against source-code provenance.
# Add columns only after documenting that they are available before prediction
# and are not used to construct the target.
PROVISIONAL_EXCLUDE = {
    TARGET,
    "risk_score",       # previously observed to perfectly separate target classes
    "transaction_type", # target-aligned pattern requires provenance review
    "purchase_pattern", # provenance not yet established
    "age_group",        # provenance not yet established
}

candidate_features = [c for c in df.columns if c not in PROVISIONAL_EXCLUDE]
print("Provisional exclusions:", sorted(PROVISIONAL_EXCLUDE))
print("Candidate features:", candidate_features)

## 3. Review identifiers and post-outcome fields

In [ ]:
possible_identifiers = [c for c in df.columns if any(
    token in c.lower() for token in ["id", "hash", "address", "wallet", "account", "user"]
)]
print("Potential identifiers / grouping fields:", possible_identifiers)
print("Review whether records from the same wallet, user, or transaction family can cross splits.")

## 4. Timestamp-based split recommendation

In [ ]:
if "timestamp" in df.columns:
    temp = df.copy()
    temp["_parsed_timestamp"] = pd.to_datetime(temp["timestamp"], errors="coerce", utc=True)
    print("Rows with invalid timestamps:", int(temp["_parsed_timestamp"].isna().sum()))
    print("Time range:", temp["_parsed_timestamp"].min(), "to", temp["_parsed_timestamp"].max())
    print("Chronological split is preferable when deployment predicts future transactions.")
else:
    print("No timestamp column; justify an alternative split strategy.")

## 5. Repeated-entity leakage checks

In [ ]:
for col in possible_identifiers:
    if col in df.columns:
        counts = df[col].value_counts(dropna=False)
        print(f"{col}: {len(counts)} unique values; {int((counts > 1).sum())} values repeated")

## 6. Preprocessing blueprint (do not fit until feature decisions are approved)

Use a scikit-learn `Pipeline` and `ColumnTransformer` so imputers, encoders, and scalers are fitted on training data only. Never preprocess the full dataset before splitting.

## 7. Required provenance register

For every candidate feature, record:
- definition and unit;
- source system/table;
- timestamp when it becomes available;
- whether it is computed from the target, a risk score, or a post-transaction outcome;
- whether it is an identifier or proxy for identity;
- keep/exclude decision and rationale.